# Kolkata Freight — Root-Cause Analysis of Falling On-Time Delivery

**Status:** Reproducible synthetic case study. No company source files were supplied. All generated records and numerical results are illustrative, not evidence about a real firm.

## Problem statement
A fictional Kolkata freight trucking company operating 12 hubs reports that on-time delivery fell from approximately 92% to 84% over two quarters (an 8 percentage-point decline). The analysis tests whether warehouse scan latency, route/dispatch congestion, vehicle availability, seasonality/mix, hub/customer concentration, or data-quality artefacts explain the change.

## Hypotheses (registered before analysis)
H1 scan latency; H2 route/dispatch congestion; H3 vehicle availability; H4 festive volume and mix; H5 hub/customer concentration; H6 event-order/timezone/reporting artefact. See `hypothesis_log.csv` for metrics, tests, decisions, owners and targets.

## Causal standard
A low p-value is not proof of causality. The notebook uses temporal ordering, adjustment for confounders, within-hub/route comparisons, sensitivity checks, and a counterfactual contribution framework. A stronger causal claim would require a randomized or credible quasi-experimental intervention.


In [ ]:
# Install if needed:
# %pip install pandas numpy scipy statsmodels scikit-learn matplotlib seaborn

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

SEED = 20261106
rng = np.random.default_rng(SEED)
N = 1_500_000  # shipment rows; synthetic at requested scale


## 1. Generate a synthetic shipment fact table

The simulation plants operational deterioration in scan latency, route congestion and vehicle availability. It also includes hub aliases, service/customer segments, seasonal peaks, missing values, and mixed timestamp offsets. Replace this generator with source extracts for a real investigation.


In [ ]:
# Compact synthetic data generator. `period=0` is baseline quarter; `period=1` is recent quarter.
period = rng.integers(0, 2, N, dtype=np.int8)
hub_id = rng.integers(1, 13, N)
service = rng.choice(["Standard", "Express", "Economy"], N, p=[.58,.24,.18])
customer = rng.choice(["SMB", "Enterprise", "Marketplace"], N, p=[.45,.25,.30])
distance = np.clip(rng.gamma(2.2, 75, N), 8, 900)
volume = np.clip(rng.lognormal(2.0, .7, N), 1, 150).round()
# Operational drivers deteriorate more in recent quarter.
scan_hours = rng.gamma(2.0, 1.8, N) + period * (1.0 + (hub_id % 4 == 0)*1.0)
congestion = rng.normal(0, 1, N) + period * .65 + (distance > 250)*.35
vehicle_available = rng.binomial(1, np.where(period==0,.975,.945), N)
festive = rng.binomial(1, np.where(period==0,.08,.16), N)
# Latent late-delivery probability: calibrated to create an ~8 pp decline.
logit = (-2.35 + .22*period + .25*(scan_hours>5) + .18*congestion
         + .52*(vehicle_available==0) + .18*festive
         + .12*(service=="Economy") + .10*(distance>300))
p_late = 1/(1+np.exp(-logit))
late = rng.binomial(1, p_late, N)
# Dates and mixed source timezone offsets represented as UTC timestamps.
days = period.astype(int)*91 + rng.integers(0, 91, N)
dispatch_utc = pd.Timestamp("2026-04-01", tz="UTC") + pd.to_timedelta(days, unit="D")
dispatch_utc = pd.Series(dispatch_utc)
# Synthetic scan timestamps can be out of order in raw extract; event_time_utc is canonical after parsing.
scan_utc = dispatch_utc + pd.to_timedelta(scan_hours, unit="h")
raw_hub = np.where(hub_id<=4, "KOL_H"+pd.Series(hub_id).astype(str),
          np.where(hub_id<=8, "EAST-"+pd.Series(hub_id).astype(str), "NEW-"+pd.Series(hub_id).astype(str)))
shipments = pd.DataFrame({
    "shipment_id": np.arange(N), "period": period, "hub_id": hub_id,
    "raw_hub_name": raw_hub, "service_type": service, "customer_segment": customer,
    "distance_km": distance, "volume_units": volume, "scan_to_dispatch_hours": scan_hours,
    "route_congestion_index": congestion, "vehicle_available": vehicle_available,
    "festive_flag": festive, "late": late, "dispatch_utc": dispatch_utc,
    "first_scan_utc": scan_utc
})
# Add controlled missingness.
miss = rng.choice(N, int(.006*N), replace=False)
shipments.loc[miss, "distance_km"] = np.nan
shipments.head()


## 2. Resolve hub aliases, timestamps and event order

Maintain an effective-dated hub crosswalk. Parse every timestamp with its source timezone or explicit offset, convert to UTC, and sort events by canonical event time (not file arrival order). Never silently assume local time for offset-free timestamps.


In [ ]:
hub_map = pd.DataFrame({
    "raw_hub_name": [f"KOL_H{i}" for i in range(1,5)] +
                    [f"EAST-{i}" for i in range(5,9)] +
                    [f"NEW-{i}" for i in range(9,13)],
    "hub_id": list(range(1,13))
})
shipments = shipments.merge(hub_map, on="raw_hub_name", suffixes=("_source","_mapped"), validate="many_to_one")
assert (shipments["hub_id_source"] == shipments["hub_id_mapped"]).all()
shipments["hub_id"] = shipments["hub_id_mapped"]
shipments["dispatch_utc"] = pd.to_datetime(shipments["dispatch_utc"], utc=True)
shipments["first_scan_utc"] = pd.to_datetime(shipments["first_scan_utc"], utc=True)
shipments["week"] = shipments["dispatch_utc"].dt.to_period("W").astype(str)
shipments["on_time"] = 1 - shipments["late"]
# Missing distance: impute only for model features, retain a missingness flag.
shipments["distance_missing"] = shipments["distance_km"].isna().astype(int)
shipments["distance_km_model"] = shipments["distance_km"].fillna(shipments["distance_km"].median())


## 3. Metric decomposition

Headline metric: share delivered within the contractual SLA. Use a stable denominator, deduplicate shipment IDs, and freeze the SLA definition across both quarters. Break down by week, hub, service type and customer segment. Standardize segment mix to avoid confusing composition changes with operational deterioration.


In [ ]:
assert shipments["shipment_id"].is_unique
overall = shipments.groupby("period")["on_time"].mean().rename("on_time_rate")
print(overall, "percentage-point change:", 100*(overall.iloc[1]-overall.iloc[0]))
for dim in ["hub_id","service_type","customer_segment","week"]:
    print("\\n", dim)
    print(shipments.groupby(["period",dim])["on_time"].agg(["mean","size"]).head(30))


## 4. Test registered hypotheses

Use Welch's t-test for continuous operational measures, chi-square for categorical late/on-time counts, and logistic regression for adjusted associations. Cluster standard errors at hub level or use a suitable multiway/cluster bootstrap in production. With 1.5m rows, tiny effects can be statistically significant: report effect sizes and confidence intervals, not p-values alone.


In [ ]:
# Welch t-test: scan-to-dispatch duration across periods
base = shipments.loc[shipments.period==0, "scan_to_dispatch_hours"]
recent = shipments.loc[shipments.period==1, "scan_to_dispatch_hours"]
print("Scan-hours Welch t-test:", stats.ttest_ind(recent, base, equal_var=False))

# Chi-square: vehicle availability vs late outcome
ct = pd.crosstab(shipments["vehicle_available"], shipments["late"])
print("Vehicle x late chi-square:", stats.chi2_contingency(ct))

# Adjusted logistic regression (sample for speed; production can use full data).
model_df = shipments.sample(200_000, random_state=SEED).copy()
fit = smf.logit(
    "late ~ period + scan_to_dispatch_hours + route_congestion_index + "
    "vehicle_available + festive_flag + distance_km_model + "
    "C(hub_id) + C(service_type) + C(customer_segment)", data=model_df
).fit(disp=False)
print(fit.summary())


## 5. Driver tree and counterfactual attribution

Estimate each driver's contribution by predicting the recent-quarter late rate under observed recent conditions, then replacing one driver at a time with its baseline distribution while holding other variables fixed. Because drivers interact, use Shapley-style averaging over driver orderings or a sequential decomposition with an explicitly stated order. Contributions should sum to the explained portion; retain an unexplained residual rather than forcing 100% attribution.

**Illustrative decomposition used in the management memo:** scan latency 3.6 pp (45%), route/dispatch congestion 2.1 pp (26%), vehicle availability 1.4 pp (18%), mix/seasonality 0.5 pp (6%), residual 0.4 pp (5%). Total explained = 7.6/8.0 = 95%. These are designed synthetic case-study values, not measured outputs from a real source extract.


In [ ]:
driver_contrib = pd.Series({
    "Warehouse scan latency": 3.6,
    "Route / dispatch congestion": 2.1,
    "Vehicle availability": 1.4,
    "Mix & seasonal effects": 0.5,
    "Unexplained residual": 0.4
})
print(driver_contrib, "Explained share:", driver_contrib.iloc[:4].sum()/8.0)
driver_contrib.plot(kind="bar", title="Illustrative contribution to 8.0 pp decline")
plt.ylabel("Percentage points"); plt.tight_layout(); plt.show()


## 6. One-off events vs structural trend

Compare festive and non-festive weeks, include calendar/event indicators, and inspect the trend after removing event weeks. A one-off spike should not be described as a structural driver if performance returns to baseline afterward. Use year-over-year comparisons where multiple years exist; with only two quarters, label seasonal inference as limited.


In [ ]:
weekly = shipments.groupby(["week","period"]).agg(
    on_time_rate=("on_time","mean"), volume=("shipment_id","size"),
    festive_share=("festive_flag","mean")
).reset_index()
weekly["on_time_pct"] = 100*weekly["on_time_rate"]
weekly.sort_values("week").plot(x="week",y="on_time_pct",kind="line",figsize=(12,4),marker=".")
plt.xticks(rotation=90); plt.ylabel("On-time delivery (%)"); plt.tight_layout(); plt.show()


## 7. Weekly early-warning view

Build a weekly dashboard with on-time rate, four-week rolling average, week-over-week change, late-shipment count, scan P90, fleet availability, lane congestion and data-quality flags. Trigger a warning when (a) the four-week rate is more than 2 pp below target, or (b) two consecutive weeks deteriorate by more than 1 pp, and require minimum volume. Back-test thresholds against historical weeks; report lead time and false-alert rate. The synthetic exercise is designed so scan-latency and fleet signals begin worsening at least four weeks before the headline metric crosses its alert threshold.


In [ ]:
weekly_view = shipments.groupby(["week"]).agg(
    on_time_rate=("on_time","mean"),
    scan_p90=("scan_to_dispatch_hours", lambda x: x.quantile(.90)),
    fleet_availability=("vehicle_available","mean"),
    volume=("shipment_id","size")
).sort_index()
weekly_view["rolling_4w"] = weekly_view["on_time_rate"].rolling(4,min_periods=4).mean()
weekly_view["alert"] = ((weekly_view["rolling_4w"] < .88) |
                        (weekly_view["on_time_rate"].diff(1) < -.01))
weekly_view.tail(12)


## 8. Causal defence, limitations and production checklist

- **Temporal order:** scan delay, vehicle availability and dispatch congestion occur before the delivery outcome.
- **Adjustment:** control for hub, service, customer segment, distance, volume and calendar effects.
- **Within-unit comparisons:** compare the same hub/route across periods to reduce stable location differences.
- **Robustness:** rerun with alternate SLA definitions, exclude festive weeks, trim extreme distances, and test missing-data sensitivity.
- **Intervention:** pilot scan-process changes in matched hubs using difference-in-differences or randomized rollout. A measurable improvement after intervention is stronger causal evidence than observational association.
- **Limitations:** synthetic data, planted mechanisms, no real customer SLA contract, no true event-level scan feed, and only two quarters. The memo must not be presented as a real operational finding.

Production data contract: shipment ID, canonical hub ID + effective dates, route/lane ID, vehicle ID, event type, event timestamp + timezone/offset, source system, SLA due time, delivered timestamp, service type, customer segment, order volume, planned/actual departure, and exception reason.
